## Your first pod — `kubectl run`

The Kubernetes "hello world":

```bash
kubectl run hello --image=nginx:alpine
```

That one command does a surprising amount. Under the hood:

1. `kubectl` builds a **Pod object** in memory — a small document saying "a pod named `hello` whose only container is `nginx:alpine`."
2. It `POST`s that object to the **API server** over HTTPS.
3. The API server validates it and writes it to **`etcd`**.
4. The **scheduler** notices a pod with no node assigned, picks a node (our only one in `kind`), and records that decision through the API server.
5. The **kubelet** on that node sees a new pod to run and tells the container runtime to pull `nginx:alpine` and start it.
6. The container starts. The kubelet updates status — `Pending` → `ContainerCreating` → `Running` — flowing back into `etcd`.
7. `kubectl get pods` reads that latest status from the API server.

You did steps 1 and 2. Everything else happened on its own.

Notice what `kubectl run` did *not* do: it did not SSH to a server and launch a container. It **wrote an object to a database and went home**. The rest is a *controller* watching that database and reacting. This is the heart of Kubernetes — **declarative state plus controllers that reconcile reality to it.** You'll see this exact pattern with every object in the course.

On our map, this command travels the spine left to right: the **kubectl** `run` verb → the **api-server** → down through the control loop → and finally a running **Pod** on the worker node.